### <mark><u>_**Configure the number of cores for the initial Load, with incremental 2 is enough**_</u></mark>

In [ ]:
#%%configure
#{"vCores": 16}

In [ ]:
!pip install duckrun --upgrade

### <u>_**<mark>Parameters</mark>**_</u>

In [ ]:
ws                    = 'duckrun'
lh                    = 'data'
schema                = 'aemo'
nbr_days_download     =  2
sql_folder            = 'https://github.com/djouallah/fabric_demo/raw/refs/heads/main/transformation/'
bim_url               = "https://raw.githubusercontent.com/djouallah/fabric_demo/refs/heads/main/semantic_model/directlake.bim"
remaining_files       =  max(0,nbr_days_download - 60)

In [ ]:
import duckrun
from   psutil import *
core              = cpu_count()
Nbr_threads = core * 2
print(core)

# Update Data

In [ ]:
con = duckrun.connect(f"{ws}/{lh}.lakehouse/{schema}", sql_folder)

In [ ]:
nightly =[
              
              ('scrapingv2', (["https://nemweb.com.au/Reports/Current/Daily_Reports/"],["Reports/Current/Daily_Reports/"],
                             nbr_days_download,ws,lh,Nbr_threads)),
              ('price','append'),
              ('scada','append'),
              ('download_excel',("raw/", ws,lh)),
              ('duid','overwrite'),
              ('calendar','ignore'),
              ('mstdatetime','ignore'),
              ('summary__backfill','overwrite')
         ]

intraday = [
              ('scrapingv2', (["http://nemweb.com.au/Reports/Current/DispatchIS_Reports/","http://nemweb.com.au/Reports/Current/Dispatch_SCADA/" ],
                            ["Reports/Current/DispatchIS_Reports/","Reports/Current/Dispatch_SCADA/"],
                             288, ws,lh,Nbr_threads)),
              ('price_today','append'),
              ('scada_today','append'),
              ('duid','ignore'),
              ('summary__incremental', 'append')            
          ]

history_download = [('scrapingv2',(["https://github.com/djouallah/fabric_demo/tree/main/data/archive/*"],["Reports/Current/Daily_Reports/"],
                                   remaining_files,ws,lh,Nbr_threads))]

history_process = [('scada','append'),('price','append'),('summary__backfill_archive','append')]

In [8]:
#create lakehouse if not exists
con.create_lakehouse_if_not_exists(lh)

OK Lakehouse 'data' created successfully


True

In [9]:
%%time
con.run(nightly)


Task 1/8: scrapingv2
Running Python: scrapingv2(['https://nemweb.com.au/Reports/Current/Daily_Reports/'], ['Reports/Current/Daily_Reports/'], 2, 'duckrun', 'data', 16)
Could not read log file Reports/Current/Daily_Reports/download_log.csv: Object at location data.Lakehouse/Files/Reports/Current/Daily_Reports/download_log.csv not found: Error performing GET https://onelake.blob.fabric.microsoft.com/duckrun/data.Lakehouse/Files/Reports/Current/Daily_Reports/download_log.csv in 3.0513348s - Server returned non-2xx status code: 404 Not Found: {"error":{"code":"PathNotFound","message":"The specified path does not exist.\nRequestId:1836dd88-601f-00ab-2383-835d88000000\nTime:2026-01-12T05:25:49.2460707Z"}}

Debug source:
NotFound {
    path: "data.Lakehouse/Files/Reports/Current/Daily_Reports/download_log.csv",
    source: RetryError(
        RetryErrorImpl {
            method: GET,
            uri: Some(
                https://onelake.blob.fabric.microsoft.com/duckrun/data.Lakehouse/Files

True

In [10]:
%%time
con.run(intraday)


Task 1/5: scrapingv2
Running Python: scrapingv2(['http://nemweb.com.au/Reports/Current/DispatchIS_Reports/', 'http://nemweb.com.au/Reports/Current/Dispatch_SCADA/'], ['Reports/Current/DispatchIS_Reports/', 'Reports/Current/Dispatch_SCADA/'], 288, 'duckrun', 'data', 16)
Could not read log file Reports/Current/DispatchIS_Reports/download_log.csv: Object at location data.Lakehouse/Files/Reports/Current/DispatchIS_Reports/download_log.csv not found: Error performing GET https://onelake.blob.fabric.microsoft.com/duckrun/data.Lakehouse/Files/Reports/Current/DispatchIS_Reports/download_log.csv in 2.748004s - Server returned non-2xx status code: 404 Not Found: {"error":{"code":"PathNotFound","message":"The specified path does not exist.\nRequestId:db492a6e-301f-009b-4c84-83b8c4000000\nTime:2026-01-12T05:29:49.5335969Z"}}

Debug source:
NotFound {
    path: "data.Lakehouse/Files/Reports/Current/DispatchIS_Reports/download_log.csv",
    source: RetryError(
        RetryErrorImpl {
            m

True

In [11]:
%%time
con.deploy(bim_url)


Semantic Model Deployment (DirectLake)
✅ Using cached Fabric API token

[Step 1/6] Getting workspace information...
OK Found workspace: duckrun

[Step 2/6] Checking if dataset 'aemo' exists...
OK Dataset name 'aemo' is available

[Step 3/6] Finding lakehouse...
OK Found lakehouse: data

[Step 3.5/6] Resolving to GUIDs for semantic model...
OK Workspace GUID: 5efa9e49-7783-41f0-b15a-61dc23c40c6d
OK Item GUID: 0ad8a9ea-fb82-43f1-9b43-12b53f914c63

[Step 4/6] Loading and configuring BIM file...
OK BIM file downloaded from URL
  - Tables: 4
  - Relationships: 3
OK Updated BIM for DirectLake
  - OneLake URL: https://onelake.dfs.fabric.microsoft.com/5efa9e49-7783-41f0-b15a-61dc23c40c6d/0ad8a9ea-fb82-43f1-9b43-12b53f914c63
  - Schema: aemo

[Step 5/6] Deploying semantic model...
OK Semantic model created
   Waiting for operation to complete...
OK Operation completed
   Dataset ID: 1f6cf253-e32f-488a-964a-7598c0aa049c
   Waiting 5 seconds before refresh...

[Step 6/6] Refreshing semantic model

1

In [12]:
%%time
if remaining_files > 0:
    con.run(history_download)

CPU times: total: 0 ns
Wall time: 0 ns


In [13]:
%%time
if remaining_files > 0:
    con.run(history_process)

CPU times: total: 0 ns
Wall time: 0 ns


In [14]:
totalrows=(con.sql("select count(*) from summary").fetchone()[0])
print(totalrows)

133860
